# Advanced Spark SQL & UDFs

#### Learning objectives
- How to work with Structs and Arrays
- Creating and applying User Defined Functions (UDFs)
- (Exploring the stars...)

### Installing Spark

In [1]:
#Checking the installed Java version
!java -version

openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu124.04.1, mixed mode, sharing)


In [2]:
!pip install pyspark 

In [3]:
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless


Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:2 https://download.docker.com/linux/ubuntu noble InRelease                 
Hit:3 https://cli.github.com/packages stable InRelease                         
Hit:4 https://security.ubuntu.com/ubuntu noble-security InRelease              
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease  
Hit:7 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 http://deb.wakemeops.com/wakemeops stable InRelease                      
Hit:9 https://cloud.archive.ubuntu.com/ubuntu noble InRelease                  
Hit:10 https://archive.ubuntu.com/ubuntu noble InRelease
Hit:11 https://cloud.archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:12 https://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:13 https://cloud.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:14 https://archive.ubuntu.

In [4]:
!java -version

openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu124.04.1, mixed mode, sharing)


In [5]:

# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local[*]")\
        .appName("Planetary") \
        .getOrCreate()
print("Spark ready:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/21 14:54:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark ready: 4.0.1


In [7]:
# Importing functions and types
from pyspark.sql import types as tp


#### Reading the planets dataset

In [8]:
# Define the file path
file_path = "/teamspace/studios/this_studio/week07/lab7_bda_solved.csv"



In [9]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, ArrayType
)

# --- 0) Read CSV with robust quote handling (prevents column shifting) ---
csv_schema = "uuid string, system string, stardate_discovery string, flybys string, habitability_index string, elements string, planet_taxonomy string, planetary_temperatures string"

df = (spark.read
      .option("header", "true")
      .option("multiLine", "true")                     # if any rows span lines
      .option("quote", "\"")
      .option("escape", "\"")
      .option("unescapedQuoteHandling", "BACK_TO_DELIMITER")  # key for messy quotes
      .schema(csv_schema)
      .csv(file_path))

# --- 1) Define target nested schemas ---
flybys_schema = ArrayType(ArrayType(StringType()))
elements_schema = ArrayType(StringType())
planet_taxonomy_schema = StructType([
    StructField("taxonomy_level1", StringType(), True),
    StructField("taxonomy_level2", StringType(), True),
    StructField("confidence_level", DoubleType(), True),
])
planetary_temps_schema = ArrayType(
    StructType([
        StructField("measurement8",  DoubleType(), True),
        StructField("measurement4",  DoubleType(), True),
        StructField("measurement5",  DoubleType(), True),
        StructField("measurement3",  DoubleType(), True),
        StructField("measurement7",  DoubleType(), True),
        StructField("measurement1",  DoubleType(), True),
        StructField("measurement9",  DoubleType(), True),
        StructField("type",          StringType(), True),
        StructField("measurement2",  DoubleType(), True),
        StructField("measurement6",  DoubleType(), True),
        StructField("measurement10", DoubleType(), True),
    ])
)

# --- 2) Minimal normaliser: trim + collapse doubled quotes introduced by CSV ---
def normalize_json(col):
    c = F.trim(col)
    # Turn [""ABC""] -> ["ABC"], and remove stray trailing/leading quotes/brackets artifacts
    c = F.regexp_replace(c, r'""', '"')
    return c

# --- 3) Safely coerce the numeric column (handles values like ["0.87"] or bad tokens) ---
NUM = r'([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)'
habitability_num = F.expr("try_cast(habitability_index as double)")
habitability_fallback = F.expr(f"try_cast(regexp_extract(habitability_index, '{NUM}', 1) as double)")
habitability_final = F.coalesce(habitability_num, habitability_fallback)

# --- 4) Parse JSON-ish columns into proper types ---
planetary_df = (
    df
    .withColumn("habitability_index", habitability_final)
    .withColumn("flybys", F.from_json(normalize_json("flybys"), flybys_schema))
    .withColumn("elements", F.from_json(normalize_json("elements"), elements_schema))
    .withColumn("planet_taxonomy", F.from_json(normalize_json("planet_taxonomy"), planet_taxonomy_schema))
    .withColumn("planetary_temperatures", F.from_json(normalize_json("planetary_temperatures"), planetary_temps_schema))
)




In [10]:
# Print the schema of the DataFrame
planetary_df.show(5)

+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+----------------------+
|                uuid|              system|  stardate_discovery|              flybys| habitability_index|            elements|     planet_taxonomy|planetary_temperatures|
+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+----------------------+
|1856a72e-2fde-4f0...|Proxima Centauri ...|4484.736272674565...|[[WXY-234], [STU-...|  0.648708376148531|[Platinum, Hafniu...|{savannah, dry, 0...|  [{30.301, 37.108,...|
|33dd1b1a-cf51-434...| HD 209458 Formation|4755.956377260051...|       [[WXY-12345]]| 0.4966529856392779|[Ytterbium, Yttri...|{arctic, frozen, ...|  [{-3.143, 1.928, ...|
|50bcbce3-1776-498...| HD 209458 Formation|1794.713080699246...|[[ZAB-6789], [TUV...|0.47792946887426335|  [Terbium, Yttrium]|{desert, dry, 0.704

In [12]:
planetary_df.printSchema()

root
 |-- uuid: string (nullable = true)
 |-- system: string (nullable = true)
 |-- stardate_discovery: string (nullable = true)
 |-- flybys: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: string (containsNull = true)
 |-- habitability_index: double (nullable = true)
 |-- elements: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- planet_taxonomy: struct (nullable = true)
 |    |-- taxonomy_level1: string (nullable = true)
 |    |-- taxonomy_level2: string (nullable = true)
 |    |-- confidence_level: double (nullable = true)
 |-- planetary_temperatures: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- measurement8: double (nullable = true)
 |    |    |-- measurement4: double (nullable = true)
 |    |    |-- measurement5: double (nullable = true)
 |    |    |-- measurement3: double (nullable = true)
 |    |    |-- measurement7: double (nullable = true)
 |    |    |-- measure

Note that the column `planet_taxonomy` is a complex data structure: an `struct` with the information of each taxonomy element.

In PySpark, a complex data structure refers to a data type that can store multiple values, often with varying data types or nested structures. Complex data structures in PySpark include:  
1. `Arrays`: An array is an ordered collection of elements, where each element can be of any data type, including other complex data types. In PySpark, arrays are represented using the ArrayType class.  

2. `Structs`: A struct is a collection of named fields, where each field can have a different data type, including other complex data types. Structs are similar to rows in a table or objects in a programming language. In PySpark, structs are represented using the StructType class.  

3. `Maps`: A map is a collection of key-value pairs, where keys are unique and both keys and values can be of any data type, including other complex data types. Maps are useful for representing associative arrays, dictionaries, or hash maps. In PySpark, maps are represented using the MapType class.  

These complex data structures allow you to work with more sophisticated data in your PySpark applications, such as nested JSON data or hierarchical data. You can manipulate complex data structures using built-in PySpark functions, as well as user-defined functions (UDFs) when necessary.

#### Reading messy CSV into nested columns (what the code is doing, step-by-step)

**Why this is needed.** Our CSV stores *nested* data (arrays and structs) as text that looks like JSON, but the quotes are messy (e.g., `[""ABC""]`). If we read it naively, columns can shift or the JSON fails to parse. So we: **read safely → clean strings → parse into real nested types → fix numbers**.

**1) Safe CSV read (keep columns aligned).**
We pass a simple **top-level schema** (strings for the JSON-ish columns) and enable a few CSV options to tolerate bad quotes and multiline rows. This prevents “column drift” during parsing and skips slow type inference. Spark’s CSV reader supports options like `quote`, `escape`, `multiLine`, and `unescapedQuoteHandling` (e.g., `BACK_TO_DELIMITER` or `STOP_AT_CLOSING_QUOTE`) to handle unescaped quotes robustly. 

**2) Normalise the JSON-ish strings.**
Before we can parse, we make the text into **valid JSON**. We use `trim` and `regexp_replace` to collapse doubled quotes (`""` → `"`). These are standard column expressions in PySpark and run lazily as part of the DataFrame plan. 

**3) Parse strings into real nested columns.**
With clean JSON text, we call `from_json(col, schema)` and supply the **exact nested schema** we want (e.g., `array<array<string>>` for `flybys`, a `StructType` with named fields for `planet_taxonomy`, and an array of structs for `planetary_temperatures`). `from_json` materialises true Spark types (`array`, `struct`) and returns `NULL` for rows that still can’t be parsed. 

**4) Safely coerce the numeric column.**
Casting directly with ANSI semantics will fail on malformed values (e.g., `["0.87"]`). We therefore use `try_cast` to return `NULL` instead of throwing, and as a fallback we `regexp_extract` the first numeric token from the string (handles wrapped numbers like `["0.49665"]`) and cast that. This is the recommended pattern under ANSI mode. 

**5) Putting it together with `withColumn`.**
Each transformation (`from_json`, `try_cast`, regex cleanup) is applied via `withColumn`, which either creates or replaces a column with a new expression. Chaining these steps produces a DataFrame whose schema now has real **arrays** and **structs** that you can `select`, `explode`, aggregate, or save to Parquet/Delta. 

**Key takeaways.**

* CSV is **flat**; treat nested content as strings first, then parse to nested Spark types with `from_json` and an explicit schema.
* Use CSV options to avoid column misalignment when quotes are messy; `unescapedQuoteHandling` is especially helpful.
* Under ANSI, prefer `try_cast` (and regex fallback) for resilient numeric parsing.

Once parsed, these complex columns behave like any other Spark columns: you can reach into structs (`col("planet_taxonomy.confidence_level")`), index arrays, or `explode` them for row-wise operations.


#### Step 1: Extract the first probe that inspected each planet

#### Step 2: Extract only the daytime temperatures from each row

#### Step 3: Count how many rare elements there are on each planet

`transform` is one the important functions to manipulate arrays that you should know. It returns an array of elements after applying a transformation function to each element in the input array.

#### Step 4: Filter only rows whose `elements` column contains ``Ytterbium``

#### Step 5: What if we want each array element on a single row?

#### Step 5.1: What about collecting the array?

> There are dozens of functions that can be applied to array manipulation. These are the ones I find most useful, but you can take a look at the documentation and look for all functions specified as a "Collection function" in the description.

Here's a list of important array functions that you should be familiar with when working with Spark DataFrames:
1. `array()`: Create an array from multiple columns or values.
2. `concat()`: Concatenate multiple arrays.
3. `size()`: Get the size (length) of an array.
4. `element_at()`: Retrieve an element at a specific index in an array.
5. `slice()`: Extract a subarray from an array.
6. `array_contains()`: Check if an array contains a specific value.
7. `arrays_zip()`: Combine multiple arrays into a single array of structs.
8. `array_distinct()`: Remove duplicate elements from an array.
9. `array_except()`: Return an array containing elements from the first array that are not present in the second array.
10. `array_intersect()`: Return an array containing elements that are present in both input arrays.
11. `array_union()`: Return an array containing elements from both input arrays, without duplicates.
12. `array_remove()`: Remove a specific value from an array.
13. `array_sort()`:Sort the elements of an array in ascending order.
14. `array_max()`: Find the maximum value in an array.
15. `array_min()`: Find the minimum value in an array.
16. `array_position()`: Find the position (index) of a specific value in an array.
17. `array_repeat()`: Create an array by repeating a value for a specified number of times.
18. `flatten()`: Flatten a nested array structure.

#### More Flexibility: Enter UDFs!

> **Meet UDFs**

So far, we have explored many out-of-the-box functions and methods to manipulate data using PySpark. But what if we need more flexibility?  

**User Defined Functions** or **UDFs** are exactly for that. They can be used to perform specific transformation that could not be done by spark `built-in` functions.
In general, we apply `UDFs` exactly when there's a given transformation that we cannot do with a pyspark function, because they have a worse performance (specially in python).  
Thus, it's important for you to know that when you are using a UDF, you're trading performance for flexibility.

**Creating a UDF** is somewhat easy. You just need to:  

1. Create and document the function
2. Make sure the input and output types are compatible
3. Test the function
4. Register the function as a `spark udf`

Alright, now let's create a python function that calculates the size of a list by an integer and return the sum of values as an integer:

It works!  

Once you have your Python function created, PySpark provides a simple mechanism to promote to a UDF, the `udf` function, that will be used to "promote" your pure python function to an UDF.  

The function takes two parameters:

- The function you want to promote
- The return type of the generated UDF, using pySpark datatypes

Now we can apply this functions to our sample `sparkDataFrame`:

#### Step 6: Create a UDF to perform the operation of extracting the daytime temperature measurements as a list

Now, let's compare the performance of the UDF against the built-in PySpark functions

PySpark User-Defined Functions (UDFs) are useful when you need to perform a specific operation on your data that is not easily achievable using built-in PySpark functions.   

**However, it's important to note that using UDFs can sometimes lead to performance issues, as they require data serialization between the JVM and Python processes. Therefore, it's recommended to use built-in functions whenever possible and only resort to UDFs whennecessary.**  

Here are some scenarios when you should consider using PySpark UDFs:
1. **Custom transformations**: When you need to apply a custom transformation to a column or multiple columns in your DataFrame that cannot be achieved using built-in functions.
2. **Complex calculations**: When you have to perform complex calculations or operations on your data that are not available in the built-in functions library.
3. **Domain-specific logic**: When you need to implement domain-specific logic or business rules that are unique to your use case and not covered by built-in functions.
4. **Integration with external libraries**: When you want to leverage external Python libraries in your PySpark code, you can wrap the library functions in a UDF to use them in your DataFrame operations.
However, it's important to note that using UDFs can sometimes lead to performance issues, as they require data serialization between the JVM and Python processes. Therefore, it's recommended to use built-in functions whenever possible and only resort to UDFs whennecessary.

#### What you know is your best friend: Pandas UDFs in Spark

![pandas_udf_performance](https://databricks.com/wp-content/uploads/2017/10/image1-4.png)

Instead of `udf()`, we are going to use `pandas_udf()`, again, from the `pyspark.sql.functions module`.  
Optionally (but recommended), we can pass the return type of the UDF as an argument to the `pandas_udf()` decorator.  

Our function signature is also different: rather than using scalar values (such as int or str), the UDF takes pd.Series and return a pd.Series.

![pandas_udf](https://drek4537l1klr.cloudfront.net/rioux/Figures/09-02.png)

Another way of applying pandas UDFs is after a groupby. But in this case, this is a dataframe to dataframe UDF, which means that your UDF will receive a dataframe and must output another dataframe:

How this `.applyInPandas` works?

Your grouped data can be transformed using groupBy().applyInPandas() to implement the “split-apply-combine” pattern. Split-apply-combine consists of three steps:

- Split the data into groups by using DataFrame.groupBy.

- Apply a function on each group. The input and output of the function are both pandas.DataFrame. The input data contains all the rows and columns for each group.

- Combine the results into a new DataFrame.


Let's use this split-apply-combine PandasUDF to compute the average habitability index for each type of planetary environment within each star system

#### Step 7: What is the most habitable star system overall?